***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft

pd.options.display.float_format = '{:.1f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'EIA')
    path_main = os.path.join(path_sp, 'Data')

path_code    = os.path.join(path_git, 'Data', 'EIA')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://www.eia.gov/opendata/documentation.php
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

***

Preparing API Request

***

In [ ]:
# Execute script to prepare API request inputs
exec(open(os.path.join(path_code, 'config', 'Configuration File.py')).read())

***

Processing

***

In [ ]:
cat         = df_api['category'   ].values[0]
route1      = df_api['route1'     ].values[0]
route2      = df_api['route2'     ].values[0]
facetOption = df_api['facetOption'].values[0]
facet       = df_api['facet'      ].values[0]
freq        = freq.lower()


if indicator_name == 'Road_1':
    end = 'raw.csv'

    df_road1_us = pd.read_csv(os.path.join(path_raw, f"{indicator_name}_EIA_{cat}_{route1}_{route2}_{facetOption}_NUS_{freq}_{end}"))
    df_road1_ca = pd.read_csv(os.path.join(path_raw, f"{indicator_name}_EIA_{cat}_{route1}_{route2}_{facetOption}_SCA_{freq}_{end}"))

    df_road1_us = df_road1_us[df_road1_us['product-name'] == 'Regular Gasoline']
    df_road1_ca = df_road1_ca[df_road1_ca['product-name'] == 'Regular Gasoline']

    df_eia = pd.concat([df_road1_us, df_road1_ca])
    
    df_eia = df_eia.dropna()
    df_eia = df_eia[df_eia['value'] != 'null']
    
    df_eia = df_eia.sort_values(['period', 'duoarea'], ascending = [False, True])
    df_eia = df_eia.reset_index(drop = True)
    df_eia = df_eia[['period', 'area-name', 'product', 'product-name', 'process-name', 'value']]
    df_eia.columns = ['Year', 'Geography', 'Product', 'Product Name', 'Process', 'Price']
    df_eia['Price'] = df_eia['Price'].astype('float32')
    df_eia['Year' ] = df_eia['Year' ].astype('int')
    df_eia = df_eia[df_eia['Year'] >= 2000]

    df_eia.loc[df_eia['Geography'] == 'U.S.'      , 'Geography'] = 'National'
    df_eia.loc[df_eia['Geography'] == 'CALIFORNIA', 'Geography'] = 'California'
    
    display(df_eia.head())

***

Exporting

***

In [ ]:
year_start = 2001
year_end = 2023
sample_type = 'EIA'
tag = 'Gas Prices'

df_about = write_about(sample_type      = sample_type
                       , indicator_name = indicator_name
                       , year_start     = year_start
                       , year_end       = year_end
                       , path_config0   = path_config0)

df_about

In [ ]:
df_indicators = pd.read_excel(os.path.join(path_config, 'Indicators.xlsx'), sheet_name = 'Indicators')
df_indicators = df_indicators[df_indicators['Indicator'] == indicator_name]

export_loc = df_indicators['Export Location'].values[0]

path_out = os.path.join(path_main, export_loc)

workbook_name = f'{indicator_name} {tag} {sample_type}.xlsx'

with pd.ExcelWriter(os.path.join(path_out, workbook_name), engine='xlsxwriter') as writer:
    df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
    df_eia  .to_excel(writer, index = False, sheet_name = tag                  )

print('Export Location: ' + path_out)


In [ ]:
# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data"

workbook_name = f'{indicator_name} {tag} {sample_type}.xlsx'

with pd.ExcelWriter(os.path.join(path_plots, workbook_name), engine='xlsxwriter') as writer:
    df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
    df_eia  .to_excel(writer, index = False, sheet_name = tag                  )

print('Export Location: ' + path_plots)
